# Notebook 03: AST XGBoost baseline

This notebook reproduces the first Phase 3 AST baseline flow using the real training and evaluation scripts. It loads the manifest, inspects extracted AST/dataflow features, trains the baseline via `scripts/train_ast_baseline.py`, compares against `majority_baseline.json`, and reads `/tmp/rtl_phase3_models/report.json`. Current metrics are bootstrap-pipeline validation only, not final quality claims.

In [8]:
from pathlib import Path
import json
import shutil
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'rtl_analyzer').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from notebook working directory')


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def repo_relative(path: Path) -> str:
    try:
        return path.relative_to(repo_root).as_posix()
    except ValueError:
        return path.as_posix()


def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        args,
        cwd=repo_root,
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout or f'command failed: {args}')
    return result


print(repo_relative(repo_root))

.


## Load the manifest generated by the dataset pipeline

Notebook 03 assumes the deterministic bootstrap dataset exists at `/tmp/rtl_phase3_dataset/manifest.json`. If needed, it rebuilds that dataset first with the same production script used in Notebook 02.

In [9]:
from rtl_analyzer.ml.dataset_manifest import read_manifest

dataset_root = Path('/tmp/rtl_phase3_dataset')
manifest_path = dataset_root / 'manifest.json'
build_script_rel = Path('scripts/build_phase3_dataset.py')
build_script = repo_root / build_script_rel
build_script_hint = 'scripts/build_phase3_dataset.py'
if not manifest_path.exists():
    rebuild_args = [
        sys.executable,
        str(build_script),
        '--output-dir',
        str(dataset_root),
        '--seed',
        '7',
        '--synthetic-source',
        str(repo_root / 'tests' / 'fixtures' / 'buggy'),
        '--external-source',
        str(repo_root / 'tests' / 'fixtures' / 'clean'),
    ]
    rebuild = run_command(rebuild_args)
    print(rebuild.stdout.strip())
manifest = read_manifest(manifest_path)
print({'manifest': manifest_path.as_posix(), 'entries': len(manifest.entries), 'seed': manifest.seed})

{'manifest': '/tmp/rtl_phase3_dataset/manifest.json', 'entries': 20, 'seed': 7}


## Extract AST features from real manifest entries

The baseline does not use hand-written notebook features. It relies on `rtl_analyzer/ml/ast_features.py`, which combines AST counts with the shared dataflow summary.

In [10]:
from rtl_analyzer.ml.ast_features import extract_ast_features
from rtl_analyzer.parser import parse_file

feature_rows = []
for entry in manifest.entries[:3]:
    parsed = parse_file(manifest_path.parent / entry.path)
    feature_rows.append({
        'sample_id': entry.sample_id,
        'split': entry.split,
        'path': entry.path,
        'features': extract_ast_features(parsed),
    })
print(json.dumps(feature_rows, indent=2, sort_keys=True))

[
  {
    "features": {
      "always_block_count": 2.0,
      "always_comb_count": 0.0,
      "always_ff_count": 2.0,
      "assign_count": 0.0,
      "dataflow_cycle_count": 0.0,
      "dataflow_error": 0.0,
      "dataflow_node_count": 0.0,
      "line_count": 24.0,
      "module_count": 1.0,
      "parse_error_count": 0.0
    },
    "path": "train/synthetic/buggy_cdc.v",
    "sample_id": "synthetic:buggy_cdc.v",
    "split": "train"
  },
  {
    "features": {
      "always_block_count": 0.0,
      "always_comb_count": 0.0,
      "always_ff_count": 0.0,
      "assign_count": 3.0,
      "dataflow_cycle_count": 1.0,
      "dataflow_error": 0.0,
      "dataflow_node_count": 3.0,
      "line_count": 7.0,
      "module_count": 1.0,
      "parse_error_count": 0.0
    },
    "path": "train/synthetic/buggy_combo_loop.v",
    "sample_id": "synthetic:buggy_combo_loop.v",
    "split": "train"
  },
  {
    "features": {
      "always_block_count": 3.0,
      "always_comb_count": 1.0,
      "alw

In [11]:
train_script_rel = Path('scripts/train_ast_baseline.py')
xgboost_status = 'XGBoost remains optional here; inspect the backend field to see whether xgboost or the sklearn fallback ran.'
print({'train_script': train_script_rel.as_posix(), 'note': xgboost_status})


{'train_script': 'scripts/train_ast_baseline.py', 'note': 'XGBoost remains optional here; inspect the backend field to see whether xgboost or the sklearn fallback ran.'}


## Train the AST baseline with the production script

`scripts/train_ast_baseline.py` selects the available backend. The Phase 3 code keeps XGBoost explicit rather than silently required: if `xgboost` is installed and enabled in the classifier implementation, artifacts record the `xgboost` backend; otherwise the stable scikit-learn fallback is used. Either way, this notebook exercises the real training path and writes the expected Task 6 artifacts.

In [12]:
models_root = Path('/tmp/rtl_phase3_models')
train_script = repo_root / 'scripts' / 'train_ast_baseline.py'
predictions_path = models_root / 'ast_predictions.jsonl'
metrics_path = models_root / 'ast_metrics.json'
baseline_path = models_root / 'majority_baseline.json'
train_args = [
    sys.executable,
    str(train_script),
    '--manifest',
    str(manifest_path),
    '--model-dir',
    str(models_root),
    '--predictions-out',
    str(predictions_path),
    '--metrics-out',
    str(metrics_path),
    '--baseline-out',
    str(baseline_path),
]
train_result = run_command(train_args)
print('Command:')
print(' '.join([repo_relative(train_script)] + train_args[2:]))
print(train_result.stdout.strip() or 'training completed with artifact outputs only')

Command:
scripts/train_ast_baseline.py --manifest /tmp/rtl_phase3_dataset/manifest.json --model-dir /tmp/rtl_phase3_models --predictions-out /tmp/rtl_phase3_models/ast_predictions.jsonl --metrics-out /tmp/rtl_phase3_models/ast_metrics.json --baseline-out /tmp/rtl_phase3_models/majority_baseline.json
training completed with artifact outputs only


## Compare the AST baseline against the majority baseline

These metrics come from a tiny bootstrap dataset. Treat them as proof that the manifest, feature extraction, training, prediction, and evaluation plumbing works together, not as a final model claim.

In [13]:
metrics_payload = json.loads(metrics_path.read_text(encoding='utf-8'))
majority_payload = json.loads(baseline_path.read_text(encoding='utf-8'))
prediction_lines = [json.loads(line) for line in predictions_path.read_text(encoding='utf-8').splitlines() if line.strip()]
comparison = {
    'backend': metrics_payload['backend'],
    'held_out_split': metrics_payload['held_out_split'],
    'held_out_count': metrics_payload['held_out_count'],
    'ast_metrics': metrics_payload['metrics'],
    'majority_metrics': majority_payload['metrics'],
    'ast_predictions_preview': prediction_lines[:3],
}
print(json.dumps(comparison, indent=2, sort_keys=True))

{
  "ast_metrics": {
    "accuracy": 0.6666666666666666,
    "macro_f1": 0.4
  },
  "ast_predictions_preview": [
    {
      "backend": "sklearn_random_forest",
      "path": "test/synthetic/buggy_counter.v",
      "predicted_label": "buggy",
      "probabilities": {
        "buggy": 0.75,
        "clean": 0.25
      },
      "sample_id": "synthetic:buggy_counter.v",
      "split": "test",
      "true_label": "buggy"
    },
    {
      "backend": "sklearn_random_forest",
      "path": "test/synthetic/buggy_fsm.sv",
      "predicted_label": "buggy",
      "probabilities": {
        "buggy": 0.640625,
        "clean": 0.359375
      },
      "sample_id": "synthetic:buggy_fsm.sv",
      "split": "test",
      "true_label": "buggy"
    },
    {
      "backend": "sklearn_random_forest",
      "path": "test/external/clean_latch_default_at_top.v",
      "predicted_label": "buggy",
      "probabilities": {
        "buggy": 0.859375,
        "clean": 0.140625
      },
      "sample_id": "extern

## Read the aggregate evaluation report

The training script writes model and baseline artifacts. The evaluation step writes `/tmp/rtl_phase3_models/report.json`, which is the portable summary to inspect or hand off.

In [14]:
evaluate_script = repo_root / 'scripts' / 'evaluate_phase3.py'
report_path = models_root / 'report.json'
evaluate_args = [
    sys.executable,
    str(evaluate_script),
    '--manifest',
    str(manifest_path),
    '--predictions',
    str(predictions_path),
    '--report-out',
    str(report_path),
]
evaluate_result = run_command(evaluate_args)
print(evaluate_result.stdout.strip() or 'evaluation completed with report output only')
report_payload = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report_payload, indent=2, sort_keys=True))

evaluation completed with report output only
{
  "ast_baseline": {
    "backend": "sklearn_random_forest",
    "metrics": {
      "accuracy": 0.6666666666666666,
      "macro_f1": 0.4
    }
  },
  "held_out_count": 3,
  "held_out_split": "test",
  "integrity": {
    "extra_prediction_ids": [],
    "matched_prediction_ids": [
      "external:clean_latch_default_at_top.v",
      "synthetic:buggy_counter.v",
      "synthetic:buggy_fsm.sv"
    ],
    "missing_prediction_ids": []
  },
  "majority_baseline": {
    "label": "buggy",
    "metrics": {
      "accuracy": 0.6666666666666666,
      "macro_f1": 0.4
    }
  },
  "note": "This Phase 3 report is based on a tiny bootstrap dataset and is suitable for pipeline validation, not broad performance claims."
}


## Rerun paths

Re-run training and evaluation manually:

```bash
python scripts/train_ast_baseline.py --manifest /tmp/rtl_phase3_dataset/manifest.json --model-dir /tmp/rtl_phase3_models --predictions-out /tmp/rtl_phase3_models/ast_predictions.jsonl --metrics-out /tmp/rtl_phase3_models/ast_metrics.json --baseline-out /tmp/rtl_phase3_models/majority_baseline.json
python scripts/evaluate_phase3.py --manifest /tmp/rtl_phase3_dataset/manifest.json --predictions /tmp/rtl_phase3_models/ast_predictions.jsonl --report-out /tmp/rtl_phase3_models/report.json
```

Re-execute this notebook and the focused tests:

```bash
python -m jupyter nbconvert --to notebook --execute --inplace notebooks/rtl_analyzer_phase3/03_ast_xgboost_baseline.ipynb
python -m pytest tests/test_phase3_ml.py::test_ast_classifier_reports_backend_and_probabilities -v
python -m pytest tests/test_phase3_docs.py::test_notebook_03_exists_and_mentions_xgboost_and_metrics -v
```

The key production files for this path are `rtl_analyzer/ml/ast_features.py`, `rtl_analyzer/ml/classifiers.py`, `scripts/train_ast_baseline.py`, and `scripts/evaluate_phase3.py`.